DSCI 552 Homework 3

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

from itertools import combinations
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re

Question 1.b
- Keep datasets 1 and 2 in folders bending1 and bending 2
- Keep datasets 1, 2, and 3 in other folders as test data and other datasets as train data

In [9]:
# all folders -> lines 1-5 are headers, line 6 starts the data, separated by ","
path_to_AReM = Path("../data/AReM")
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

training_dfs = {}
testing_dfs = {}

for data_dir in path_to_AReM.iterdir():
    if data_dir.is_dir(): # want to skip the .pdfs
        dir_name = data_dir.name
            
        for file in data_dir.iterdir():
            file_name = Path(file).stem
            file_num = int("".join(re.findall(r"\d", file_name))) # extracting the number to put in training or testing folder
            # ERROR TESTING !!!!
            # bad_count = 0
            # max_print = 5
            # with open(file, "r") as f:
            #     for i, line in enumerate(f):
            #         if i < 5:
            #             continue
            #         parts = line.strip().split(",")
            #         if len(parts) != len(column_names):
            #             print(f"BAD LINE -> {dir_name}, {file_name}, {line}")
            #             bad_count += 1
            # print(bad_count)
            # FOUND:
                # bending2 dataset 4 -> delimiter is " ", not "," -> fixing in the code
                # cycling dataset 9, 14 -> last line has trailing "," -> opened in TextEdit and removed the trailing ","

            key = f"{dir_name}_{file_name}" # bending1_dataset1 for example
            
            if (dir_name == "bending1") or (dir_name == "bending2"):
                # datasets 1, 2 = test. rest = train
                if dir_name == "bending1":
                    df = pd.read_csv(file, skiprows=5, names=column_names)
                    # first 5 rows are not data, adding the column names saved above
                else:
                    df = pd.read_csv(file, skiprows=5, names=column_names, sep=" ")
                    # this file has a different delimiter 
                
                if (file_num == 1) or (file_num == 2):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df
            else: 
                # datasets 1, 2, 3, = test. rest = train
                df = pd.read_csv(file, skiprows=5, names=column_names)
                
                if (file_num == 1) or (file_num == 2) or (file_num == 3):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df
          

Question 1.c
- Feature Extraction
- Classification of time series usually needs extracting features from them
- In this problem, we focus on time-domain features

Question 1.c.i
- Research what types of time-domain features are usually used in time series classification and list them (examples are minimum, maximum, mean, etc.)

1. Mean
2. Median
3. Mode
4. Min
5. Max
6. Range
7. Variance
8. Standard Deviation
9. Skewness (describes assymetry. 0 = symmetric. + = right-skewed. - = left-skewed)
10. Kurtosis (describes shape of a data distribution, quantifying how often extreme outliers occur)(tails)
11. Autocorrelation (measures relationship between a varibale and its lagged values / tells how much a data point is influenced by its own past values)
12. Zero Crossing Rate (rate at which signal changes signs)
14. Trend (a sequence of values recorded over time, used to analyize patterns / monitor changes / make forecasts)
15. Seasonality (recurring / regular patterns at a set interval)

Question 1.c.ii
- Extract the time-domain features:
- - minimum, maximum, mean, median, standard deviation, first quartile, third quartile
- for all of the 6 time series in each instance (the column names saved earlier)
- Free to normalize / standardize features or use them directly (can experiment to see if it makes a difference)
- New dataset will look like:
- - Instance (1 -> 88), Min1, Max1, Mean1, Median1, ... 1st quart6, 3rd quart6
  - (1st quart6 = first quartile of the SIXTH time series in each of the 88 instances)

For each dataset in each folder: -> 42 columns for each new dataset
1. avg_rss12 -> min, max, mean, median, std dev, 1Q, 3Q
2. var_rss12 -> min, max, mean, median, std dev, 1Q, 3Q
3. avg_rss13 -> min, max, mean, median, std dev, 1Q, 3Q
4. var_rss13 -> min, max, mean, median, std dev, 1Q, 3Q
5. avg_rss23 -> min, max, mean, median, std dev, 1Q, 3Q
6. var_rss23 -> min, max, mean, median, std dev, 1Q, 3Q

In [10]:
columns_six = column_names[1:] # not using "time"

training_features = [] # will by 69 rows x 42 columns
# want it to look like [{dirfilename : name, min1 : x, max1 : x, ...}, {}, ...]

for dirfile1, its_df1 in training_dfs.items():
    a_instance = {}
    a_instance["dirfile_name"] = dirfile1 # to keep track of where the numbers are coming from

    for i, a_column in enumerate(columns_six, start=1):
        a_instance[f"min_{i}"] = its_df1[a_column].min()
        a_instance[f"max_{i}"] = its_df1[a_column].max()
        a_instance[f"mean_{i}"] = its_df1[a_column].mean()
        a_instance[f"median_{i}"] = its_df1[a_column].median()
        a_instance[f"std_dev_{i}"] = its_df1[a_column].std()
        a_instance[f"first_quart_{i}"] = its_df1[a_column].quantile(0.25)
        a_instance[f"third_quart_{i}"] = its_df1[a_column].quantile(0.75)

    training_features.append(a_instance)

In [11]:
testing_features = [] # will by 19 rows x 42 columns

for dirfile2, its_df2 in testing_dfs.items():
    b_instance = {}
    b_instance["dirfile_name"] = dirfile2

    for j, b_column in enumerate(columns_six, start=1):
        b_instance[f"min_{j}"] = its_df2[b_column].min()
        b_instance[f"max_{j}"] = its_df2[b_column].max()
        b_instance[f"mean_{j}"] = its_df2[b_column].mean()
        b_instance[f"median_{j}"] = its_df2[b_column].median()
        b_instance[f"std_dev_{j}"] = its_df2[b_column].std()
        b_instance[f"first_quart_{j}"] = its_df2[b_column].quantile(0.25)
        b_instance[f"third_quart_{j}"] = its_df2[b_column].quantile(0.75)

    testing_features.append(b_instance)

Question 1.c.iii
- Estimate the standard deviation of each of the time-domain features you extracted from the data
- Then, use Python's boostrapped or any other method to build a 90% bootstrap confidence interval for the standard deviation of each feature

In [12]:
all_features = training_features + testing_features
all_features_df = pd.DataFrame(all_features)
af_df = all_features_df.drop(columns=["dirfile_name"]) # don't need dirfilename for this part
std_of_features = af_df.std()

In [13]:
# Boostrap Part
    # per column -> resample with replacement the 88 values in that column ->
    # calculate std dev each time -> use distribution
        # Need to calc> std dev of the column (non bootstrap, just the actual std dev)
                      # boostrap resample to make new distribution
                      # left 5% tail
                      # right 5% tail
boostrap_results = []
# list of dictionaries again
# looks like: [{"feat" : "min1", "std_dev" : x, "left_tail" : x, "right_tail" : x}, {}, ...]
for a_column in af_df.columns:
    one_feat = {}
    one_feat["feat"] = a_column
    one_feat["std_dev"] = std_of_features[a_column]

    # boostrap calculating now
    boostrap_samples = []

    for _ in range(1000): # _ -> don't care what iteration we're on, just do it 1000 times
        boostrap_sample = af_df[a_column].sample(frac=1.0, replace=True)
        # frac=1.0 -> randomly select and return 100% of the available data
        boostrap_sample_std = boostrap_sample.std()
        boostrap_samples.append(boostrap_sample_std)

    bootstrap_series = pd.Series(boostrap_samples) # Series -> 1D labeled array

    left_tail = bootstrap_series.quantile(0.05)
    right_tail = bootstrap_series.quantile(0.95)

    one_feat["left_tail"] = left_tail
    one_feat["right_tail"] = right_tail

    boostrap_results.append(one_feat)

In [14]:
for result in boostrap_results:
    for key, value in result.items():
        print(f"{key} -> {value}")
    print("\n")

feat -> min_1
std_dev -> 9.56326483288447
left_tail -> 8.114620417933255
right_tail -> 10.856994427147432


feat -> max_1
std_dev -> 6.457254618060868
left_tail -> 3.5059818536520324
right_tail -> 9.261078099606458


feat -> mean_1
std_dev -> 6.4232034795914075
left_tail -> 4.355079504148417
right_tail -> 8.754798500987478


feat -> median_1
std_dev -> 6.490874078071817
left_tail -> 4.432845362048715
right_tail -> 8.714705965184603


feat -> std_dev_1
std_dev -> 1.70571579718889
left_tail -> 1.526379651933553
right_tail -> 1.8751361635139179


feat -> first_quart_1
std_dev -> 6.944297579264584
left_tail -> 5.233925575487391
right_tail -> 8.880969076219085


feat -> third_quart_1
std_dev -> 6.391175747493661
left_tail -> 3.999177533181037
right_tail -> 8.925528844474776


feat -> min_2
std_dev -> 0.0
left_tail -> 0.0
right_tail -> 0.0


feat -> max_2
std_dev -> 5.241813838039777
left_tail -> 4.797123435350508
right_tail -> 5.586006870458809


feat -> mean_2
std_dev -> 1.7544425529198162

Question 1.c.iv
- Use your judgement to select the three most important time-domain features

- What I am using to decide:
- - If the std_dev is high -> it actually varies over time, may show patterns with time / more info
  - If the width of the confidence interval is small -> means results are predictable / consistent output / fewer outliers
- (I put the above print out into a note to better see and compare the outputs)

THREE FEATURES I'M SELECTING:
- Max
- - Has the highest standard deviations with tight confidence interval widths, so I think this one is the strongest
- Median
- - I was trying to decide between Median, Mean, and Q1, but Median has slightly higher standard deviations with decently tight CI widths as well, so that's why I chose it over the other two
- Third Quartile
- - Has high standard deviations with tight CI widths

Question 2, ISLR 3.7.4
- I collect a set of data (n = 100 observations) containing a single predictor and a quantitative response
- I then fit a linear regression model to the data, as well as a separate cubic regression (ie: Y = B0 + B1X + B2X^2 + B3X^3 + epsilon)
- (a) Suppose that the true relationship between X and Y is linear (ie: Y = B0 + B1X + epsilon)
- - Consider the training residual sum of squares (RSS) for the linear regression, and also the training RSS for the cubic regression
  - Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer
- (b) Answer (a) using test rather than training RSS.
- (c) Suppose that the true relationship between X and Y is not linear, but we don't know how far it is from linear
- - Consider the training RSS for the linear regression, and also the training RSS for the cubic regression.
  - Would we expect them to be the same, or is there not enough information to tell? Justify your answer.
- (d) Answer (c) using test rather than training RSS.

MY ANSWERS:
- (a) I think during training the cubic would do better / have a lower RSS because even though the true relationship is linear, if we consider noise then in the training stage the cubic model can better fit the noise using the extra non-linear terms. The linear model wouldn't be able to fit all the noise. Not saying this is good, but in training alone I think it would result in a better RSS than linear.
- (b) I think the cubic model would do worse during testing because of what I said in part (a). The cubic model would have most likely fit more noise than the linear model in training, so on new unseen data it would most likely do worse because it's too complicated for the true linear relationship.
- (c) During training, I think that the cubic model would do better than the linear model still. This is because it once again has more flexibility than the linear model to fit whatever relationship it might be (especially since we know it's non-linear).
- (d) For testing, I am inclined to say that the cubic would still do better than the linear model, but I think in this case there is not enough information. My thought is: what if it is super non-linear, like a sine wave? Or what if it's exponential? In that case I think both would do bad (again, I would MAYBE think cubic would do better), but I can't say for sure if one would have a better RSS than the other.

REFERENCES, Questions Asked on Search Engines:
- How to iterate through a folder in python using pathlib Path
- - https://www.google.com/search?q=how+to+iterate+through+a+folder+in+python+using+pathlib+Path&oq=how+to+iterate+through+a+folder+in+python+using+pathlib+Path&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIHCAEQIRigATIHCAIQIRirAjIHCAMQIRirAjIHCAQQIRiPAtIBCDkwMTZqMGo5qAIGsAIB8QU8YooxynnjWw&sourceid=chrome&ie=UTF-8
- how to extract just the numbers from the name of .csv files python
- - https://www.google.com/search?q=how+to+extract+just+the+numbers+from+the+name+of+.csv+files+python&oq=how+to+extract&gs_lcrp=EgZjaHJvbWUqCAgAEEUYJxg7MggIABBFGCcYOzIGCAEQRRg5MggIAhBFGCcYOzIGCAMQRRg7Mg0IBBAAGJECGIAEGIoFMgwIBRAAGBQYhwIYgAQyBwgGEAAYgAQyBwgHEAAYgAQyBwgIEAAYgAQyBwgJEAAYgATSAQgyMzcwajBqN6gCALACAA&sourceid=chrome&ie=UTF-8
- (ChatGPT) Error Help to fix errors in the data files
- - https://chatgpt.com/c/6a2cc4b5-232c-83e8-8d3b-fe71effb91fb
- Websites used to research time-domain features used in time series classification
- - https://medium.com/@adnan.mazraeh1993/time-series-features-from-basics-to-super-advanced-3589944f6a98
- - https://stats.stackexchange.com/questions/50807/features-for-time-series-classification
- - https://www.geeksforgeeks.org/data-analysis/feature-engineering-for-time-series-data-methods-and-applications/
- Autocorrelation Definition
- - https://www.geeksforgeeks.org/machine-learning/autocorrelation/
- Kurtosis and Skewness Definition
- - https://www.geeksforgeeks.org/data-science/difference-between-skewness-and-kurtosis/
- Zero Crossing Rate Definition
- - https://openae.io/standards/features/latest/zero-crossing-rate/
- Trend Defintion
- - https://www.geeksforgeeks.org/python/what-is-a-trend-in-time-series/
- Seasonality Definiton
- - https://www.geeksforgeeks.org/machine-learning/seasonality-detection-in-time-series-data/
- how to iterate through a list while adding numbers associated with the index starting at 1 python
- - https://www.google.com/search?q=how+to+iterate+through+a+list+while+adding+numbers+associated+with+the+index+starting+at+1+python&sca_esv=48277157c5478a00&sxsrf=ANbL-n6SQxJOmxVBQY4WSioZvclSR6C3Ow%3A1781382786040&ei=gr4taoCNAoCn0PEP0crL-Ag&ved=0ahUKEwiA1q6BiIWVAxWAEzQIHVHlEo8Q4dUDCBI&uact=5&oq=how+to+iterate+through+a+list+while+adding+numbers+associated+with+the+index+starting+at+1+python&gs_lp=Egxnd3Mtd2l6LXNlcnAiYWhvdyB0byBpdGVyYXRlIHRocm91Z2ggYSBsaXN0IHdoaWxlIGFkZGluZyBudW1iZXJzIGFzc29jaWF0ZWQgd2l0aCB0aGUgaW5kZXggc3RhcnRpbmcgYXQgMSBweXRob25IAFAAWABwAHgAkAEAmAEAoAEAqgEAuAEDyAEA-AEBmAIAoAIAmAMAkgcAoAcAsgcAuAcAwgcAyAcAgAgB&sclient=gws-wiz-serp
- how to get standard deviation of each time domain feature of a dataframe python
- - https://www.google.com/search?q=how+to+get+standard+deviation+of+each+time+domain+feature+of+a+dataframe+python&oq=how+to+get+standard+deviation+of+each+time+domain+feature+of+a+dataframe+python&gs_lcrp=EgZjaHJvbWUyBggAEEUYOdIBCTEyNTQxajBqN6gCALACAA&sourceid=chrome&ie=UTF-8
- how to do boostrap confidence intervals in python with pandas
- - https://www.google.com/search?q=how+to+do+boostrap+confidence+intervals+in+python+with+pandas&sca_esv=e7d38744cd64d9bd&sxsrf=ANbL-n4xgn2bs58B2DhjWK8_1rbnWVJ2QA%3A1781402044515&ei=vAkuavX5HriH0PEPpYSYoAU&biw=638&bih=787&ved=0ahUKEwj1x8Lgz4WVAxW4AzQIHSUCBlQQ4dUDCBI&uact=5&oq=how+to+do+boostrap+confidence+intervals+in+python+with+pandas&gs_lp=Egxnd3Mtd2l6LXNlcnAiPWhvdyB0byBkbyBib29zdHJhcCBjb25maWRlbmNlIGludGVydmFscyBpbiBweXRob24gd2l0aCBwYW5kYXNIAFAAWABwAHgBkAEAmAEAoAEAqgEAuAEDyAEA-AEBmAIAoAIAmAMA4gMFEgExIECSBwCgBwCyBwC4BwDCBwDIBwCACAE&sclient=gws-wiz-serp
- pandas dataframe properties
- - https://www.google.com/search?q=pandas+dataframe+properties&oq=pandas+dataframe+properties&gs_lcrp=EgZjaHJvbWUqBwgAEAAYgAQyBwgAEAAYgAQyCAgBEAAYFhgeMggIAhAAGBYYHjIICAMQABgWGB4yCggEEAAYChgWGB4yCAgFEAAYFhgeMg0IBhAAGIYDGIAEGIoFMgcIBxAAGO8FMgcICBAAGO8FMgoICRAAGKIEGIkF0gEINDY1M2owajeoAgCwAgA&sourceid=chrome&ie=UTF-8
- how many iterations is common for bootstrap resampling
- - https://www.google.com/search?q=how+many+iterations+is+common+for+boostrap+resampling&oq=how+many+iterations+is+common+for+boostrap+resampling&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIJCAEQIRgKGKABMgkIAhAhGAoYoAEyCQgDECEYChigATIJCAQQIRgKGKABMgcIBRAhGJ8FMgcIBhAhGI8C0gEINzg4MmowajeoAgCwAgA&sourceid=chrome&ie=UTF-8
- what does pd.Series do
- - https://www.google.com/search?q=what+does+pd.Series+do&oq=what+does+pd.Series+do&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIICAEQABgWGB4yCAgCEAAYFhgeMg0IAxAAGIYDGIAEGIoFMg0IBBAAGIYDGIAEGIoFMg0IBRAAGIYDGIAEGIoFMgcIBhAAGO8FMgcIBxAAGO8F0gEIMjYzMmowajeoAgCwAgA&sourceid=chrome&ie=UTF-8
- for a distribution of standard deviations, is it better the left and right tails to be further apart or closer together? (I was trying to wrap my head around what this all meant)
- - https://www.google.com/search?q=for+a+distribution+of+standard+deviations%2C+is+it+better+the+left+and+right+tails+to+be+further+apart+or+closer+together%3F&oq=for+a+distribution+of+standard+deviations%2C+is+it+better+the+left+and+right+tails+to+be+further+apart+or+closer+together%3F&gs_lcrp=EgZjaHJvbWUyBggAEEUYOdIBCTcyMzkzajBqOagCBrACAfEFqEGNxplEFG8&sourceid=chrome&ie=UTF-8 